# 📓 MapBiomas Fire Monitor — Pipeline (M0-M7)


## 📘 Documentación y Guía del Usuario
Expanda esta sección para entender la arquitectura del dato, reglas de nomenclatura y fluxo de trabalho.

#### 📌 Introducción y Contexto de Uso

Este pipeline constituye el núcleo de procesamiento automatizado para el **Monitoramiento de Cicatrices de Fuego de MapBiomas**. Su objetivo principal es extraer, procesar, clasificar y publicar datos satelitales regionalizados.

*   📥 **Entradas:** Colecciones Earth Engine (S2, Landsat), vectores de entrenamiento y mapas auxiliares (LULC).
*   📤 **Salidas:** Chunks y Mosaicos COG en Google Cloud Storage (GCS), Modelos DNN y ImageCollections pre-oficiales en GEE.

#### 🔄 **Recomendación de Entorno de Ejecución**

*   🚀 **Google Colab (Recomendado):** Sugerimos utilizar el Colab para **todas las etapas**. Ofrece un entorno pre-configurado, acceso directo a recursos de la nube y una velocidad de red superior para el tráfico con GCS y GEE.
*   💻 **Ejecución Local (Alternativa):** Es posible ejecutar el pipeline localmente, pero requiere que el usuario gestione manualmente las dependências (GDAL, Conda, Python) y realice adaptaciones específicas según su sistema operativo (Windows ou Linux).

| Complejidad Local | Módulos | Requisito Principal |
| :--- | :--- | :--- |
| **🔵 Leve** | M0, M3, M7 | Autenticación GEE/GCS |
| **🟡 Media** | M1, M2, M4 | GDAL Binaries + Disco SSD |
| **🔴 Pesada** | M5, M6 | GDAL Binaries + Python VirtualEnv |

> **Nota:** Independiente del entorno, la persistencia de datos ocurre siempre de forma centralizada en el **Google Cloud Storage (GCS)**.

#### 🚦 Ciclo de Vida del Dato y Reglas de Nomenclatura

| Etapa | Peso | Inputs → Outputs | Regla de Nomenclatura | Ejemplo |
| :--- | :--- | :--- | :--- | :--- |
| **M1: Export** | **Leve** | **IN:** Colecciones GEE<br>**OUT:** Chunks GCS | `[sensor]_fire_[país]_[año][mes]` | `s2_fire_peru_2408` |
| **M2: Mosaic** | **Medio** | **IN:** Chunks GCS<br>**OUT:** Mosaicos COG (GCS) | Semilla M1 + sufijo banda | `s2_fire_peru_2408_nir_cog.tif` |
| **M3: Toolkit** | **Leve** | **IN:** Mosaicos (M2)<br>**OUT:** Polígonos (GCS/Asset) | `[col]_[sat_ref]_[reg]`| `samples_sentinel2_r01` |
| **M4: Train** | **Medio** | **IN:** Muestras M3 + Mosaicos M2<br>**OUT:** DNN (GCS) | `[ver]_[sensor]_[reg]` | `v1_1_sentinel2_r01` |
| **M5: Classify**| **Pesado**| **IN:** Modelo M4 + Mosaicos M2<br>**OUT:** Raster (GCS) | `klass_[país]_[reg]_[mod]_[yymm]`| `klass_peru_r01_v1_2408` |
| **M6: Filters** | **Leve** | **IN:** DOY M5 + LULC<br>**OUT:** Raster Filtrado (GCS)| `filt_[país]_[reg]_[mod]_[yymm]`| `filt_peru_r01_v1_2408` |
| **M7: Versioner**| **Leve** | **IN:** Candidatos M6<br>**OUT:** Asset PRE-OFICIAL | `[colección]_v[X]` | `peru_fire_col1_v1` |


### 📂 Arquitectura de Datos e Relacionamento (M1-M7)

O monitor opera um fluxo circular de sincronização entre três ambientes:

| Ambiente | Componentes Principais | Papel no Ciclo |
| :--- | :--- | :--- |
| **🌍 Google Earth Engine** | ImageCollections (Mosaicos) / Assets | Fonte de Brutos e Destino Final |
| **☁️ Google Cloud Storage** | Chunks / COGs / Models / Samples | Persistência e Área de Trabalho Central |
| **⚡ Cache Local (Temp)** | VRT Stack / Tiles / NumPy Arrays | Processamento I/O (Local: HD | Colab: /content) |

#### 🧭 Mapa de Persistência (Onde encontrar os dados)

| Etapa | Extensão | Path Principal no Cloud Storage (GCS) | 
| :--- | :--- | :--- |
| **M1: Export** | `.tif` | `library_images/{sensor}/monthly/chunks/{yyyy}/{mm}/` |
| **M2: Mosaic** | `.tif` | `library_images/{sensor}/monthly/cog/{yyyy}/{mm}/` |
| **M3: Samples** | `.shp` | `rawsamples/{anual,monthly}/{ano}/` |
| **M4: Train** | `.pb / .json` | `library_images/{sensor}/model/{model_name}/` |
| **M5: Classify** | `.tif` | `library_images/{sensor}/monthly/classifications/raw_versions/v{v}/` |
| **M7: Public** | Asset IC | `projects/mapbiomas-public/assets/{country}/fire/monitor/` |

#### 🏷️ Regras de Nomenclatura Padrão
*   **Fragmentos (M1):** `{sensor}_fire_{pais}_{yy}{mm}_{banda}.tif`
*   **Mosaico COG (M2):** `{sensor}_fire_{pais}_{yy}{mm}_{banda}_cog.tif`
*   **Classificação (M5):** `class_{sensor}_{pais}_{modelo}_{yymm}_v{v}.tif`

---

#### 🔄 Retroalimentación y Tolerancia a Fallos

*   **Restantes:** El botón "Seleccionar Restantes" marcará solo los ítems **pendientes** en GCS.
*   **Sobrescritura:** Remarcar manualmente un ítem con bandera de éxito (verde) indica la intención de reemplazar el archivo.
*   **Modo Edición (`EDIT_MODE`):** Si es activado en `M0`, la UI expone botones de "Eliminar" en GCS.


## ⚙️ [M0] — Configuración de Ambiente (Escolha uma Rota)


### > Opción A: Inicialización Google Colab


In [ ]:
# M0.1a — Preparación del entorno Colab (Clonar repo)
import os

# 💡 DICA: Si el repositorio es privado, usa: https://<TOKEN>@github.com/mapbiomas/peru-fire.git
REPO_URL = "https://github.com/mapbiomas/peru-fire.git"

if not os.path.exists("peru-fire"):
    !git clone {REPO_URL}

# Ajuste de ruta basado en el repositorio oficial de MapBiomas Peru
%cd peru-fire/mapbiomas_fire_monitor/version_01/src/core


#### > Instalación de Dependencias Vitales (GDAL)
El monitor utiliza utilitarios de sistema de **GDAL** para ensamblar mosaicos COG de alto rendimiento. Esta celda debe ejecutarse una vez por sesión en Colab.

In [ ]:
# ⬇️ Instalar GDAL Binaries e dependências Python
!apt-get update -qq && apt-get install -y -qq gdal-bin python3-gdal
!pip install -q earthengine-api gcsfs rasterio scipy tqdm


In [ ]:
# Autenticación en Google Cloud / GEE
from google.colab import auth
auth.authenticate_user()


### > Opción B: Inicialización Local


**🛠️ Requisitos Local (GDAL / Conda)**

Si recibes un error de `Faltam dependências vitais` ejecutando localmente, asegúrate de tener os binários do GDAL instalados.
*   **Conda:** `conda install -c conda-forge gdal` (Recomendado)
*   **Nota:** `pip install gdal` no es suficiente para as herramientas de mosaico.


In [ ]:
# M0.1b — Configuración local de rutas
import sys, os
REPO_ROOT = os.path.abspath("..")
SRC_PATH  = os.path.join(REPO_ROOT, "src", "core")
if SRC_PATH not in sys.path: sys.path.insert(0, SRC_PATH)

# Requisito Local: GDAL (gdalbuildvrt, gdal_translate) deve estar no PATH
# Dica: Use 'conda install -c conda-forge gdal' ou OSGeo4W no Windows
# En consola: (1) earthengine authenticate (2) gcloud auth application-default login


## 🚀 Rota Sequencial (Fluxo Padrão de Processamento)


### > [M0.2] — Inicialización del Monitor


In [ ]:
# M0.2 — Inicialización del Monitor
import sys, os

# ─── CONFIGURACIÓN DE PARÁMETROS CRÍTICOS ───────────────────────────
COUNTRY     = "peru"       # peru, bolivia, paraguay
SENSOR      = "sentinel2"  # sentinel2, landsat, hls, modis
PERIODICITY = "monthly"    # monthly, yearly
EDIT_MODE   = False        # True → habilita herramientas de edición GCS
# ──────────────────────────────────────────────────────────────────

def auto_path_setup():
    """Localiza a pasta src/core em diferentes ambientes"""
    possible_paths = [
        os.path.abspath("."),             # Já está na pasta?
        os.path.abspath("../src/core"),   # Estrutura local padrão
        "/content/peru-fire/mapbiomas_fire_monitor/version_01/src/core", # Colab
    ]
    for p in possible_paths:
        if os.path.exists(os.path.join(p, "M0_auth_config.py")):
            if p not in sys.path: sys.path.insert(0, p)
            return p
    return None

found_path = auto_path_setup()

try:
    from M0_auth_config import set_country, authenticate, print_config, set_global_opts, set_edit_mode
    set_country(COUNTRY)
    set_global_opts(sensor=SENSOR, periodicity=PERIODICITY)
    set_edit_mode(EDIT_MODE)
    authenticate()
    print_config()
except ImportError as e:
    print(f"❌ ERROR: No se detectaron los scripts: {e}")


### [M1] — Despacho de Exportación (GEE → Bucket)

**Peso:** 🔵 Leve | **Acción:** Selección tabular y despacho de tareas GEE.


In [ ]:
import importlib
import M1_export_ui
importlib.reload(M1_export_ui)

ui_exporter = M1_export_ui.run_ui(years=[2025,2026])


In [ ]:
from M1_export_ui import start_export
start_export(ui_exporter)


### [M2] — Ensamblaje Nacional (COG)

**Peso:** 🟡 Medio | **Acción:** Unión de fragmentos mediante GDAL (VRT -> COG).


In [ ]:
import importlib
import M2_mosaic_ui
importlib.reload(M2_mosaic_ui)
from M2_mosaic_ui import run_ui, start_assemble

# Dispara a renderizacao segura
ui_assembler = run_ui(years=[2025,2026])


In [ ]:
from M2_mosaic_ui import start_assemble
start_assemble(ui_assembler) # Si obtiene NameError, verifique la celda anterior


### [M3.3] — Toolkit de Recolección de Samples

**Peso:** 🔵 Leve | **Acción:** Dibujo de muestras y agrupación para entrenamiento.


In [ ]:
from M3_sample_ui import run_collection_toolkit
ui_toolkit = run_collection_toolkit()


In [ ]:
from M3_sample_ui import run_grouping_ui
ui_group = run_grouping_ui()


In [ ]:
from M3_sample_ui import start_sample_extraction
start_sample_extraction(ui_group)


### [M4] — Entrenamiento del Modelo (DNN)

**Peso:** 🟡 Medio | **Acción:** Configuración de hiperparámetros y entrenamiento.


In [ ]:
from M4_model_trainer import run_ui as run_trainer
ui_trainer = run_trainer()


In [ ]:
from M4_model_trainer import start_training
start_training(ui_trainer)


### [M5] — Clasificación Regional

**Peso:** 🔴 Pesado | **Acción:** Inferencia masiva regional.


In [ ]:
# Preset de modelos por región
PRESET_MODELS = {
  "v1_deep_sentinel2_peru": ["loreto_norte", "san_martin"],
  "v2_exp_sentinel2_peru":  ["ucayali_sur", "madre_de_dios"]
}
from M5_classifier import run_ui as run_classifier
ui_classifier = run_classifier(PRESET_MODELS)


In [ ]:
from M5_classifier import start_classification
start_classification(ui_classifier)


### [M6] — Aplicación de Filtros Físicos y LULC

**Peso:** 🟢 Leve | **Acción:** Aplicación de máscaras y filtros morfológicos focalizados.


In [ ]:
# Preset de filtros por región
PRESET_FILTERS = {
    "peru_r1": {
        "mask_classes": [26, 33, 24],
        "open_filter": 3,
        "close_filter": 3
    }
}
from M6_publisher import run_ui as run_filters
ui_filters = run_filters(PRESET_FILTERS)


In [ ]:
from M6_publisher import start_filtering
start_filtering(ui_filters)


### [M7] — Curaduría y Colección Pre-Oficial

**Peso:** 🟣 Morado | **Acción:** Selección de ganadores y exportación a GEE Asset.


In [ ]:
# Preset de votos (variante ganadora por región)
PRESET_VOTES = {
    "peru_r1": "filt_peru_r1_v1_sentinel2_2408"
}
from M7_curator import run_ui as run_curator
ui_curator = run_curator(PRESET_VOTES)


In [ ]:
from M7_curator import start_curation
start_curation(ui_curator)
